In [108]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')

Data source import complete.


In [109]:
import numpy as np
import pandas as pd

In [110]:
import os
df = pd.read_csv(os.path.join(titanic_path, 'train.csv'))

In [111]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [112]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [113]:
df = df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1)

In [114]:
from sklearn.model_selection import train_test_split
train_set, val_set = train_test_split(df,test_size=0.2,random_state=42)

In [115]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(strategy='mean')
train_set['Age'] = age_imputer.fit_transform(train_set[['Age']])

val_set['Age'] = age_imputer.transform(val_set[['Age']])

embarked_imputer = SimpleImputer(strategy="most_frequent")
train_set["Embarked"] = embarked_imputer.fit_transform(train_set[["Embarked"]]).ravel()

In [89]:
train_set.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 331 to 102
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Sex       712 non-null    object 
 3   Age       712 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  712 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 50.1+ KB


In [116]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

num_cols = ["Pclass","Age","SibSp","Parch","Fare"]
cat_cols = ["Sex","Embarked"]

preprocessor = ColumnTransformer([
  ('nums',StandardScaler(),num_cols),
  ('cats',OneHotEncoder(handle_unknown="ignore"),cat_cols)
])

x_train = preprocessor.fit_transform(train_set.drop("Survived", axis=1))
x_val = preprocessor.transform(val_set.drop("Survived", axis=1))

y_train = train_set["Survived"]
y_val = val_set["Survived"]

In [117]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(x_train,y_train)
pred = model.predict(x_val)

In [118]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_val,pred))

0.8100558659217877


In [120]:

test_df = pd.read_csv(os.path.join(titanic_path, 'test.csv'))
passenger_ids = test_df['PassengerId']  # save before dropping

In [121]:

from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


X_full = preprocessor.fit_transform(df.drop("Survived", axis=1))
y_full = df["Survived"]

model = LogisticRegression()
model.fit(X_full, y_full)

# now transform test_df  imputer inside pipeline handles its NaNs automatically
X_test = preprocessor.transform(test_df)
predictions = model.predict(X_test)

In [123]:

predictions = model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": predictions
})
submission.to_csv("submission.csv", index=False)